# Hyperparameter Optimization

## Brute-Force Hyperparameter Search

* Implementing a brute-force search function that finds the best parameter combination for a given model.
* Testing the implementation on the following problems:
    - 1) A `SVM` model on the two moons problem.
    - 2) A `LinearRegression` model with Ridge regularization on the `California Housing Dataset`.
Using `sklearn`'s model implementations.


In [ ]:
svm_params = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}


ridge_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

In [ ]:
import numpy as np
from sklearn.datasets import make_moons, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from itertools import product


In [ ]:
from itertools import product
import numpy as np
from sklearn.metrics import accuracy_score, mean_squared_error

def brute_force_search(model_class, param_grid,
                       X_train, y_train,
                       X_val, y_val,
                       task="classification"):
    """
    Simple brute-force grid search over param_grid.

    Parameters
    ----------
    model_class : estimator class (e.g. SVC, Ridge)
    param_grid  : dict of lists, e.g. {'C': [0.1, 1.0], 'kernel': ['linear', 'rbf']}
    X_train, y_train : training data
    X_val, y_val     : validation data
    task : "classification" or "regression"

    Returns
    -------
    best_params : dict with best hyperparameters
    best_score  : validation score (accuracy or -MSE)
    """
    param_names = list(param_grid.keys())
    param_values = [param_grid[name] for name in param_names]

    best_score = -np.inf
    best_params = None

    for values in product(*param_values):
        params = dict(zip(param_names, values))

        # instantiate model with current params
        model = model_class(**params)

        # fit and evaluate
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        if task == "classification":
            score = accuracy_score(y_val, y_pred)
        elif task == "regression":
            # use negative MSE so that "larger is better"
            score = -mean_squared_error(y_val, y_pred)
        else:
            raise ValueError("task must be 'classification' or 'regression'")

        if score > best_score:
            best_score = score
            best_params = params

    return best_params, best_score


##  Simple TPE

* Implementing the Tree-Structured Parzen Estimator using `numpy` only.
* Finding decent hyperparameters for
    - 1) An `SVM` model on the `two moons` problem.
    - 2) A `LinearRegression` model with Ridge regularization on the `California Housing Dataset`.

In [ ]:
# Two moons dataset + SVM brute-force search

# Generate data
X, y = make_moons(noise=0.3, random_state=42)

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Hyperparameter grid from the assignment
svm_params = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Run brute-force search
best_svm_params, best_svm_score = brute_force_search(
    SVC,
    svm_params,
    X_train_scaled, y_train,
    X_val_scaled, y_val,
    task="classification"
)

print("Best SVM params:", best_svm_params)
print("Best validation accuracy:", best_svm_score)

Best SVM params: {'C': 10.0, 'kernel': 'rbf', 'gamma': 'scale'}
Best validation accuracy: 0.9333333333333333


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# California Housing dataset + Ridge brute-force search

# Load data
cal = fetch_california_housing()
X, y = cal.data, cal.target

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Hyperparameter grid from the assignment
ridge_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# Run brute-force search
best_ridge_params, best_ridge_score = brute_force_search(
    Ridge,
    ridge_params,
    X_train_scaled, y_train,
    X_val_scaled, y_val,
    task="regression"
)

print("Best Ridge params:", best_ridge_params)
print("Best validation -MSE (higher is better):", best_ridge_score)
print("Corresponding validation MSE:", -best_ridge_score)


Best Ridge params: {'alpha': 100.0, 'solver': 'svd'}
Best validation -MSE (higher is better): -0.5532661022237078
Corresponding validation MSE: 0.5532661022237078
